# Module 02 — Exercises

Twelve exercises on matrices as linear maps: multiplication from scratch, geometric
    transformations and determinants, rank and null spaces, solving systems the right way,
    least squares and the hat matrix, collinearity detection, Markov chains, Gaussian elimination,
    and a mini-project fitting a multi-feature regression with your own solver. Only NumPy (plus
    matplotlib and scikit-learn for the project) is needed. Work through them in order — later
    exercises reuse ideas from earlier ones.

**How these exercises work.** Each coding exercise has a *starter* cell (replace `# YOUR CODE HERE` with your implementation) followed by a *check* cell. Run the check cell to get an instant ✅/❌. Checks never stop the notebook, so you can run everything top-to-bottom at any time and see your score at the end. Some checks compare against a *hidden* answer (only a hash is stored), so you cannot peek at the number — think first, then verify. Conceptual questions hide their answer behind a *Show answer* toggle. Mini-projects are open-ended and come with a rubric. Difficulty is marked ★☆☆ (warm-up), ★★☆ (core), ★★★ (stretch).

### Setup and self-check helpers
Run the next two cells first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

In [ ]:
import hashlib as _hashlib
import numpy as _np

_results = {}

def _canon(value, decimals=3):
    """Canonical string form of an answer, so hashes are stable."""
    if isinstance(value, (bool, _np.bool_)):
        return "true" if value else "false"
    if isinstance(value, str):
        return " ".join(value.strip().lower().split())
    if isinstance(value, (int, _np.integer)):
        return str(int(value))
    if isinstance(value, (float, _np.floating)):
        s = f"{float(value):.{decimals}f}"
        return "0." + "0" * decimals if s in ("-0." + "0" * decimals, "-0") else s
    if isinstance(value, dict):
        return "{" + ",".join(f"{_canon(k, decimals)}:{_canon(v, decimals)}" for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))) + "}"
    try:
        arr = _np.asarray(value)
        if arr.ndim == 0:
            return _canon(arr.item(), decimals)
        return "[" + ",".join(_canon(x, decimals) for x in arr.ravel().tolist()) + "]"
    except Exception:
        return str(value)

def sha_of(value, decimals=3):
    return _hashlib.sha256(_canon(value, decimals).encode()).hexdigest()[:16]

def check(name, condition, hint=""):
    """Record and print a pass/fail mark. Never raises, so the notebook keeps running."""
    try:
        ok = bool(condition)
    except Exception:
        ok = False
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"   — hint: {hint}"))
    return ok

def check_close(name, got, expected, tol=1e-6, hint=""):
    try:
        g = _np.asarray(got, dtype=float)
        e = _np.asarray(expected, dtype=float)
        ok = g.shape == e.shape and bool(_np.allclose(g, e, atol=tol, rtol=tol))
    except Exception:
        ok = False
    return check(name, ok, hint)

def check_hash(name, value, expected_sha, decimals=3, hint=""):
    """Compare an answer against a hidden expected value (only its hash is stored)."""
    try:
        ok = value is not None and sha_of(value, decimals) == expected_sha
    except Exception:
        ok = False
    return check(name, ok, hint)

def summary():
    n_ok = sum(1 for v in _results.values() if v)
    n = len(_results)
    print(f"\nScore: {n_ok}/{n} checks passed" + ("  🎉" if n and n_ok == n else ""))
    if n and n_ok < n:
        print("Not yet passing: " + ", ".join(k for k, v in _results.items() if not v))

## Exercise 1 — The row picture and the column picture  `★☆☆`

Implement the matrix–vector product twice, once per picture:

* `matvec_rows(A, x)` — entry $i$ of the result is the dot product of **row** $i$ of `A` with `x`
  (you may use `@` between two 1-D vectors, but not `A @ x`);
* `matvec_cols(A, x)` — the result is the **linear combination of the columns** of `A`,
  $\sum_j x_j\,\mathbf{a}_j$ (use scalar-times-vector and vector addition only).

Both take a 2-D array `A` of shape `(m, n)` and a 1-D array `x` of length `n`, and return a 1-D
array of length `m`. They must agree with `A @ x` for rectangular `A` as well as square `A`.

In [ ]:
def matvec_rows(A, x):
    # YOUR CODE HERE
    return None

def matvec_cols(A, x):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
A = np.array([[1., 2.], [3., 4.]]); x = np.array([2., -1.])
B = np.array([[1., 0., 2.], [0., 1., -1.], [3., 3., 3.], [1., 1., 1.]]); z = np.array([1., 2., 3.])
try:
    got_r, got_c = matvec_rows(A, x), matvec_cols(A, x)
except Exception:
    got_r = got_c = None
check_close('row picture, 2x2 example gives (0, 2)', got_r, A @ x)
check_close('column picture, 2x2 example gives (0, 2)', got_c, A @ x)
try:
    got_r, got_c = matvec_rows(B, z), matvec_cols(B, z)
except Exception:
    got_r = got_c = None
check_close('row picture, 4x3 matrix', got_r, B @ z)
check_close('column picture, 4x3 matrix', got_c, B @ z)
check('result has shape (4,) for a 4x3 matrix', got_r is not None and np.shape(got_r) == (4,))

## Exercise 2 — Matrix multiplication with three loops  `★☆☆`

Implement `matmul_loops(A, B)` computing $C_{ij} = \sum_k A_{ik}B_{kj}$ with explicit Python
loops over $i$, $j$ and $k$ — no `@`, `np.dot`, `np.matmul`, `np.einsum` or `np.sum`.
`A` has shape `(m, p)`, `B` has shape `(p, n)`, the result has shape `(m, n)`; raise a
`ValueError` when the inner dimensions disagree. Your implementation should reproduce `A @ B`
to floating-point precision for rectangular inputs, and you will use it to confirm that
$AB \neq BA$ for the swap matrix $B = \begin{pmatrix}0&1\\1&0\end{pmatrix}$.

In [ ]:
def matmul_loops(A, B):
    m, p = A.shape
    p2, n = B.shape
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
A = np.array([[1., 2.], [3., 4.]]); S = np.array([[0., 1.], [1., 0.]])
try:
    AS, SA = matmul_loops(A, S), matmul_loops(S, A)
except Exception:
    AS = SA = None
check_close('A @ S swaps the columns of A', AS, A @ S)
check_close('S @ A swaps the rows of A', SA, S @ A)
check('A S and S A differ (non-commutative)', AS is not None and SA is not None and not np.allclose(AS, SA))
M1 = rng.standard_normal((3, 4)); M2 = rng.standard_normal((4, 2))
try:
    got = matmul_loops(M1, M2)
except Exception:
    got = None
check_close('rectangular (3x4)(4x2) matches @', got, M1 @ M2)
try:
    matmul_loops(M1, M1)   # inner dims 4 and 3 do not match
    raised = False
except ValueError:
    raised = True
except Exception:
    raised = False
check('mismatched inner dimensions raise ValueError', raised)

## Exercise 3 — Rotation, shear, scaling — and their determinants  `★☆☆`

Write four small functions returning $2\times2$ NumPy arrays / floats:

* `rotation(theta)` — counter-clockwise rotation by `theta` radians;
* `shear(k)` — horizontal shear $\begin{pmatrix}1&k\\0&1\end{pmatrix}$;
* `scaling(sx, sy)` — $\mathrm{diag}(s_x, s_y)$;
* `det2(M)` — the determinant $ad - bc$ of a $2\times2$ matrix, without `np.linalg.det`.

Then predict, before running the check: what is the determinant of
`scaling(2, 3) @ shear(1) @ rotation(np.pi/2)`? (Think "product of determinants", not
"multiply the matrices".)

In [ ]:
def rotation(theta):
    # YOUR CODE HERE
    return None

def shear(k):
    # YOUR CODE HERE
    return None

def scaling(sx, sy):
    # YOUR CODE HERE
    return None

def det2(M):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
def _try(f):
    try:
        return f()
    except Exception:
        return None
check_close('rotation by 90° sends e1 to e2', _try(lambda: rotation(np.pi / 2) @ np.array([1., 0.])), np.array([0., 1.]))
check_close('rotation(a) @ rotation(b) == rotation(a + b)', _try(lambda: rotation(0.3) @ rotation(1.1)), _try(lambda: rotation(1.4)) if _try(lambda: rotation(1.4)) is not None else np.full((2, 2), np.nan))
check_close('rotations are orthogonal: R^T R = I', _try(lambda: rotation(0.7).T @ rotation(0.7)), np.eye(2))
check_close('shear(3) maps (0,1) to (3,1)', _try(lambda: shear(3.) @ np.array([0., 1.])), np.array([3., 1.]))
check_close('det of a shear is 1 (area preserved)', _try(lambda: det2(shear(3.))), 1.0)
check_close('det2 agrees with np.linalg.det', _try(lambda: det2(np.array([[2., 5.], [1., 4.]]))), np.linalg.det(np.array([[2., 5.], [1., 4.]])))
check_close('scaling(2, 3) is diag(2, 3)', _try(lambda: scaling(2., 3.)), np.diag([2., 3.]))
d = _try(lambda: float(det2(scaling(2., 3.) @ shear(1.) @ rotation(np.pi / 2))))
check_hash('hidden: det of scaling(2,3) @ shear(1) @ rotation(90°)', d, '2063c5c8a251b11c')

## Exercise 4 — Shapes, order and the collapse of linear layers  `★☆☆`

A network has two linear layers with weight matrices $W_1 \in \mathbb{R}^{4\times3}$ and
$W_2 \in \mathbb{R}^{5\times4}$ and no activation function between them. A batch of $n$ inputs is
stored as the rows of $X \in \mathbb{R}^{n\times3}$.

1. Which of $W_1W_2$ and $W_2W_1$ is defined, and what is its shape?
2. Write the batch output as a single matrix expression in $X$, $W_1$, $W_2$ (watch the
   transposes), and give its shape.
3. Why does the absence of an activation make the second layer pointless? How many free
   parameters does the two-layer network have, and how many does the equivalent single
   layer have?
4. Give a concrete $2\times2$ example where $AB \ne BA$, and one (with $A, B \ne I$ and
   $A \ne B$) where $AB = BA$.

<details><summary><b>Show answer</b></summary>

1. Only $W_2W_1$ is defined: $(5\times4)(4\times3) = 5\times3$. $W_1W_2$ would need the inner
   dimensions $3$ and $5$ to agree.
2. Per sample $\mathbf{h} = W_2 W_1 \mathbf{x}$; with samples as rows, $H = X W_1^\top W_2^\top = X (W_2 W_1)^\top$,
   shape $(n, 5)$. The transposes appear because $(W_2W_1\mathbf{x})^\top = \mathbf{x}^\top W_1^\top W_2^\top$.
3. The composition of two linear maps is a linear map, so the network computes $\mathbf{x} \mapsto M\mathbf{x}$
   with $M = W_2W_1 \in \mathbb{R}^{5\times3}$ — nothing a single layer could not do. The two-layer version has
   $5\cdot4 + 4\cdot3 = 32$ parameters (plus biases) but only $15$ effective ones, and if the hidden width
   were smaller than 3 it would even *restrict* the maps to rank $\le$ hidden width. A nonlinearity
   between the layers breaks the collapse — that is its entire job.
4. $A = \begin{pmatrix}1&2\\3&4\end{pmatrix}$, $B = \begin{pmatrix}0&1\\1&0\end{pmatrix}$:
   $AB$ swaps the columns of $A$, $BA$ swaps its rows, and they differ. Commuting examples: any
   two diagonal matrices, e.g. $A = \mathrm{diag}(2, 3)$ and $B = \mathrm{diag}(5, 7)$; or a matrix with
   its own powers or inverse, e.g. $A$ and $A^2$; or two rotations in the plane, $R_\alpha R_\beta = R_\beta R_\alpha = R_{\alpha+\beta}$.

</details>

## Exercise 5 — Rank and null space of a given matrix  `★★☆`

Consider

$$
M = \begin{pmatrix} 1 & 2 & 3 & 4 \\ 2 & 4 & 6 & 8 \\ 1 & 0 & 1 & 0 \end{pmatrix}.
$$

1. Work out **by hand** which columns are combinations of which others, and set `rank_M` (an
   `int`), `nullity_M` (an `int`, the dimension of the null space $\mathcal{N}(M)$) and
   `left_nullity_M` (an `int`, the dimension of the null space of $M^\top$: how many independent
   combinations of the *rows* add up to the zero row). Only then confirm with
   `np.linalg.matrix_rank`.
2. Set `null_vec` to any **non-zero** NumPy array of shape `(4,)` with $M\,\mathbf{v} = \mathbf{0}$,
   read off from the column relations you found (no `scipy.linalg.null_space`).
3. Set `b_solvable` to `True` or `False`: does $M\mathbf{x} = (3, 6, 1)^\top$ have a solution?
   And `b2_solvable` for $M\mathbf{x} = (1, 1, 1)^\top$? (Hint: is the right-hand side in the
   column space? Compare rows 1 and 2.)

In [ ]:
M = np.array([[1., 2., 3., 4.],
              [2., 4., 6., 8.],
              [1., 0., 1., 0.]])

rank_M = None        # YOUR ANSWER HERE (int)
nullity_M = None     # YOUR ANSWER HERE (int)
left_nullity_M = None   # YOUR ANSWER HERE (int): dimension of the null space of M^T
null_vec = None      # YOUR CODE HERE: np.array of shape (4,), non-zero, with M @ null_vec == 0
b_solvable = None    # True / False for b = (3, 6, 1)
b2_solvable = None   # True / False for b = (1, 1, 1)

**Check your answer:**

In [ ]:
check_hash('hidden: rank of M', rank_M, 'd4735e3a265e16ee')
check_hash('hidden: nullity of M', nullity_M, 'd4735e3a265e16ee')
check_hash('hidden: dimension of the null space of M^T', left_nullity_M, '6b86b273ff34fce1')
try:
    v = np.asarray(null_vec, dtype=float)
    ok = v.shape == (4,) and np.linalg.norm(v) > 1e-9 and np.allclose(M @ v, 0.0)
except Exception:
    ok = False
check('null_vec is a non-zero vector in the null space of M', ok)
check_hash('hidden: M x = (3, 6, 1) solvable?', b_solvable, 'b5bea41b6c623f7c')
check_hash('hidden: M x = (1, 1, 1) solvable?', b2_solvable, 'fcbcf165908dd18a')

## Exercise 6 — Solve a 3×3 system (by hand, then by `solve`)  `★★☆`

Solve

$$
\begin{aligned}
2x + y - z &= 8 \\
-3x - y + 2z &= -11 \\
-2x + y + 2z &= -3
\end{aligned}
$$

First by hand with elimination (write the steps on paper — it is three rows), then in code.
Set `A6` and `b6` to the coefficient matrix and right-hand side, compute `x6 = np.linalg.solve(A6, b6)`
and `det6 = np.linalg.det(A6)`. **Do not** use `np.linalg.inv`. Finally set `n_solutions` to the
string `'one'`, `'none'` or `'infinite'` describing the solution set (which the determinant tells you).

In [ ]:
A6 = None            # YOUR CODE HERE: 3x3 coefficient matrix
b6 = None            # YOUR CODE HERE: right-hand side
x6 = None            # YOUR CODE HERE: np.linalg.solve(...)
det6 = None          # YOUR CODE HERE
n_solutions = None   # 'one', 'none' or 'infinite'

**Check your answer:**

In [ ]:
try:
    resid = float(np.linalg.norm(np.asarray(A6, float) @ np.asarray(x6, float) - np.asarray(b6, float)))
except Exception:
    resid = None
check('A6 @ x6 reproduces b6 (residual < 1e-9)', resid is not None and resid < 1e-9)
check_hash('hidden: the solution vector (x, y, z)', (np.round(np.asarray(x6, float), 6) if x6 is not None else None), '13f4a57adccd27f7')
check_hash('hidden: det(A6)', (float(det6) if det6 is not None else None), '9b225070ea9ac4db')
check_hash('hidden: number of solutions', n_solutions, '7692c3ad3540bb80')

## Exercise 7 — Least squares via the normal equations  `★★☆`

Implement `lstsq_normal(A, y)` returning the least-squares coefficients $\hat{\mathbf{x}}$ that
minimise $\lVert A\mathbf{x} - \mathbf{y}\rVert^2$, by forming and solving the normal equations
$A^\top A\,\hat{\mathbf{x}} = A^\top\mathbf{y}$ with `np.linalg.solve`. Do **not** use
`np.linalg.lstsq`, `np.linalg.inv` or `np.linalg.pinv`.

Then fit the line $y = c + m\,t$ to the four points $(0, 1), (1, 2), (2, 2), (3, 4)$ by building
the design matrix `A` with an intercept column and a `t` column; store the coefficients in
`coef_line` and the residual sum of squares $\sum_i r_i^2$ in `rss_line`. Work the $2\times2$
normal equations out by hand first and compare.

In [ ]:
def lstsq_normal(A, y):
    # YOUR CODE HERE
    return None

t = np.array([0., 1., 2., 3.])
y_pts = np.array([1., 2., 2., 4.])
A_line = None        # YOUR CODE HERE: design matrix with columns [1, t]
coef_line = None     # YOUR CODE HERE
rss_line = None      # YOUR CODE HERE: residual sum of squares

**Check your answer:**

In [ ]:
Ar = np.column_stack([np.ones(30), rng.uniform(0, 10, 30), rng.standard_normal(30)])
yr = Ar @ np.array([1.5, -0.3, 2.0]) + rng.normal(0, 0.5, 30)
try:
    got = lstsq_normal(Ar, yr)
except Exception:
    got = None
ref = np.linalg.lstsq(Ar, yr, rcond=None)[0]
check_close('agrees with np.linalg.lstsq on a 30x3 design', got, ref, tol=1e-6)
try:
    orth = np.linalg.norm(Ar.T @ (yr - Ar @ got))
except Exception:
    orth = None
check('residual is orthogonal to every column (A^T r = 0)', orth is not None and orth < 1e-8)
check_hash('hidden: (intercept, slope) of the 4-point line', (np.round(np.asarray(coef_line, float), 6) if coef_line is not None else None), '0c9a8847f388e12a')
check_hash('hidden: residual sum of squares of the 4-point line', (float(rss_line) if rss_line is not None else None), '73e647f618bcea3a')

## Exercise 8 — The hat matrix is a projection  `★★☆`

The fitted values of least squares are $\hat{\mathbf{y}} = H\mathbf{y}$ with the **hat matrix**
$H = A(A^\top A)^{-1}A^\top$.

1. Implement `hat_matrix(A)` returning $H$ for a full-column-rank design matrix `A` of shape
   `(m, n)`. Do not call `np.linalg.inv` — use `np.linalg.solve(A.T @ A, A.T)` to obtain
   $(A^\top A)^{-1}A^\top$ directly.
2. Implement `is_projection(H, tol=1e-8)` returning `True` iff `H` is symmetric *and*
   idempotent ($H^2 = H$) to within `tol`. Both conditions matter: $\begin{pmatrix}1&1\\0&0\end{pmatrix}$
   is idempotent but not symmetric (an *oblique* projection) and $\mathrm{diag}(1, 2)$ is symmetric but
   not idempotent.
3. For the 4-point design matrix of the previous exercise (columns $\mathbf{1}$ and $t = 0,1,2,3$)
   store `trace_H = np.trace(H)` and `leverages = np.diag(H)`. Before running, predict the
   trace from the number of parameters and say which observations have the most leverage.

In [ ]:
def hat_matrix(A):
    # YOUR CODE HERE
    return None

def is_projection(H, tol=1e-8):
    # YOUR CODE HERE
    return None

A_line = np.column_stack([np.ones(4), np.arange(4.)])
trace_H = None       # YOUR CODE HERE
leverages = None     # YOUR CODE HERE (np.diag of the hat matrix)

**Check your answer:**

In [ ]:
Ar = np.column_stack([np.ones(12), rng.standard_normal(12), rng.standard_normal(12)])
try:
    H = hat_matrix(Ar)
    ok_shape = H.shape == (12, 12)
    ok_proj = bool(is_projection(H))
    ok_fit = np.allclose(H @ Ar, Ar)
except Exception:
    ok_shape = ok_proj = ok_fit = False
check('hat matrix has shape (m, m)', ok_shape)
check('hat matrix is symmetric and idempotent', ok_proj)
check('H leaves the columns of A unchanged (H A = A)', ok_fit)
try:
    shear_verdict = is_projection(np.array([[1., 1.], [0., 1.]]))
except Exception:
    shear_verdict = None
check('is_projection rejects a shear matrix (and accepted the hat matrix above)', ok_proj and shear_verdict is not None and not bool(shear_verdict))
try:
    oblique_verdict = is_projection(np.array([[1., 1.], [0., 0.]]))     # idempotent, but not symmetric
except Exception:
    oblique_verdict = None
try:
    diag_verdict = is_projection(np.diag([1., 2.]))                     # symmetric, but not idempotent
except Exception:
    diag_verdict = None
check('is_projection rejects an idempotent but non-symmetric matrix', ok_proj and oblique_verdict is not None and not bool(oblique_verdict))
check('is_projection rejects a symmetric but non-idempotent matrix', ok_proj and diag_verdict is not None and not bool(diag_verdict))
check_hash('hidden: trace of the 4-point hat matrix', (float(trace_H) if trace_H is not None else None), 'ebda82fe4e8dbddb')
check_hash('hidden: leverages of the 4 points', (np.asarray(leverages, float) if leverages is not None else None), '84b0d4098a4a3d96')

## Exercise 9 — Why `solve`, not `inv`?  `★★☆`

A colleague's code computes `w = np.linalg.inv(X.T @ X) @ X.T @ y` for a regression with
$n = 10^5$ rows and $d = 2{,}000$ features, and then reports that two features whose columns are
exact unit conversions of each other "got weird coefficients around $\pm 10^{12}$".

1. Explain why `inv(A) @ b` is a poor choice even when it works: compare the flop counts and
   the error behaviour with `np.linalg.solve(A, b)`.
2. Diagnose the $\pm 10^{12}$ coefficients. What property of $X$ causes them, what would
   `np.linalg.matrix_rank(X)` report, and why does `inv` not simply raise an error?
3. Suppose the two columns are *not* exactly collinear but have correlation $0.9999$. Does the
   diagnosis change? Which quantity would you compute to quantify the risk, and name two remedies.

<details><summary><b>Show answer</b></summary>

1. Solving via LU factorisation costs about $\tfrac23 d^3$ flops; forming the inverse costs about
   $2d^3$ (an LU plus $d$ triangular solves) and then the product $A^{-1}\mathbf{b}$ adds another
   $2d^2$. Worse, each entry of $A^{-1}$ carries rounding error of relative size $\kappa(A)\varepsilon$,
   and the multiplication by $\mathbf{b}$ mixes them; the triangular solves applied directly to
   $\mathbf{b}$ are backward stable, so `solve` gives a result that is the exact solution of a
   problem perturbed by only $O(\varepsilon)$. For $d = 2{,}000$ the inverse route wastes roughly
   $3\times$ the time and some digits for nothing.
2. Two exactly proportional columns make $X$ rank deficient: `matrix_rank(X)` returns $1{,}999$
   (or fewer), and $X^\top X$ is singular in exact arithmetic. In floating point the two columns are
   proportional only up to rounding, so $X^\top X$ has a tiny but non-zero smallest eigenvalue
   (around $10^{-16}$ times the largest) rather than an exact zero; `inv` therefore does not detect
   singularity and returns entries of order $1/\lambda_{\min}$, i.e. astronomically large, with opposite
   signs on the two redundant features so that their contributions cancel. Any vector in the null
   space can be added to $\mathbf{w}$ without changing $X\mathbf{w}$, so those coefficients are not
   identifiable at all.
3. The diagnosis becomes "nearly rank deficient": $X^\top X$ is invertible but ill-conditioned. The
   quantity to compute is the condition number $\kappa(X)$ (or the variance inflation factor of
   the two columns). Two separate effects appear: the standard errors of the two coefficients are
   inflated by $\sqrt{\mathrm{VIF}} = 1/\sqrt{1 - 0.9999^2} \approx 71$, a statistical statement about
   how little the data can say about each coefficient on its own; and $\kappa(X^\top X) = \kappa(X)^2$
   means that forming $X^\top X$ squares the numerical difficulty, a statement about lost digits.
   Remedies: drop or merge one of the two
   columns; use ridge regression, which solves $(X^\top X + \lambda I)\mathbf{w} = X^\top\mathbf{y}$ and
   bounds the smallest eigenvalue below by $\lambda$; or compute the fit with `lstsq`/QR, which work on
   $X$ directly and never square the condition number — though that only fixes the numerics, not
   the identifiability.

</details>

## Exercise 10 — Detect collinear features in a data matrix  `★★☆`

The cell below builds a customer feature matrix `X_cust` with 300 rows and these 10 columns:

| index | column | index | column |
|---|---|---|---|
| 0 | intercept (all ones) | 5 | region = North (0/1) |
| 1 | income (kEUR) | 6 | region = East (0/1) |
| 2 | age | 7 | region = South (0/1) |
| 3 | tenure (years) | 8 | region = West (0/1) |
| 4 | income (EUR) | 9 | noise feature |

1. Set `rank_X = np.linalg.matrix_rank(X_cust)` and predict it *before* running: how many
   independent columns are there?
2. Implement `dependent_columns(X, tol=1e-8)` returning the sorted `list` of column indices that
   take part in an exact linear dependency — the columns on which some non-zero null-space
   vector has a non-zero entry. Hint: `np.linalg.svd(X, full_matrices=False)` returns `U, s, Vt`;
   the rows of `Vt` whose singular value `s` is $\le$ `tol * s.max()` span the null space. Treat
   entries of magnitude below `1e-6` as zero.
3. Set `drop_cols` to a list of column indices such that removing exactly those columns leaves a
   matrix of full column rank, using as few columns as possible.

In [ ]:
g = np.random.default_rng(7)
n = 300
income = g.normal(50, 10, n); age = g.uniform(20, 70, n); tenure = g.uniform(0, 30, n)
region = g.integers(0, 4, n)
X_cust = np.column_stack([np.ones(n), income, age, tenure, income * 1000,
                          np.eye(4)[region], g.normal(size=n)])

rank_X = None            # YOUR CODE HERE

def dependent_columns(X, tol=1e-8):
    # YOUR CODE HERE
    return None

drop_cols = None         # YOUR ANSWER HERE: list of column indices to drop

**Check your answer:**

In [ ]:
check_hash('hidden: rank of X_cust', rank_X, '2c624232cdd22177')
try:
    got = dependent_columns(X_cust)
    got = [int(j) for j in got]
except Exception:
    got = None
check_hash('hidden: columns involved in exact dependencies', got, '0ceaa2f98988f0b5')
try:
    full = dependent_columns(np.column_stack([np.ones(20), rng.standard_normal((20, 3))]))
    ok_full = list(full) == []
except Exception:
    ok_full = False
check('a full-rank matrix has no dependent columns', ok_full)
try:
    X_small = np.array([[1., 2., 3.], [1., 0., 1.], [2., 2., 4.], [0., 1., 1.]])   # col3 = col1 + col2
    ok_small = [int(j) for j in dependent_columns(X_small)] == [0, 1, 2]
except Exception:
    ok_small = False
check('finds c1 + c2 = c3 in a 4x3 example', ok_small)
try:
    X_red = np.delete(X_cust, list(drop_cols), axis=1)
    ok_drop = len(drop_cols) == X_cust.shape[1] - np.linalg.matrix_rank(X_cust) and np.linalg.matrix_rank(X_red) == X_red.shape[1]
except Exception:
    ok_drop = False
check('drop_cols removes the minimum number of columns and leaves full column rank', ok_drop)

## Exercise 11 — Markov chain: propagate, then find the stationary distribution  `★★★`

A subscription business classifies customers each month as **Active**, **At risk** or
**Dormant**, with monthly transition matrix (rows = current state, columns = next state)

$$
P = \begin{pmatrix} 0.70 & 0.20 & 0.10 \\ 0.20 & 0.60 & 0.20 \\ 0.30 & 0.30 & 0.40 \end{pmatrix}.
$$

1. Implement `propagate(P, pi0, k)` returning the row vector $\boldsymbol\pi_0 P^k$ (the state
   distribution after `k` months) using only `@` in a loop — no `np.linalg.matrix_power`.
2. Implement `stationary(P)` returning the stationary distribution $\boldsymbol\pi$ with
   $\boldsymbol\pi P = \boldsymbol\pi$ and $\sum_i \pi_i = 1$ **by solving a linear system** with
   `np.linalg.solve` — not by iterating and not with `np.linalg.eig`. Hint: $\boldsymbol\pi(P - I) = \mathbf{0}$
   is a homogeneous system with a one-dimensional solution set; transpose it to $(P^\top - I)\boldsymbol\pi^\top = \mathbf{0}$
   and replace one of its (redundant) equations by the normalisation $\sum_i \pi_i = 1$.
3. Store `pi_3 = propagate(P, [1, 0, 0], 3)` and `pi_inf = stationary(P)`.

In [ ]:
P = np.array([[0.70, 0.20, 0.10],
              [0.20, 0.60, 0.20],
              [0.30, 0.30, 0.40]])

def propagate(P, pi0, k):
    # YOUR CODE HERE
    return None

def stationary(P):
    # YOUR CODE HERE
    return None

pi_3 = None      # YOUR CODE HERE
pi_inf = None    # YOUR CODE HERE

**Check your answer:**

In [ ]:
try:
    got1 = propagate(P, np.array([0., 1., 0.]), 1)
except Exception:
    got1 = None
check_close('one step from "At risk" is the second row of P', got1, P[1])
check_hash('hidden: distribution after 3 months from Active', (np.asarray(pi_3, float) if pi_3 is not None else None), '96312b6fe293e9f7')
try:
    s = np.asarray(stationary(P), float)
    ok_stat = s.shape == (3,) and np.allclose(s @ P, s) and abs(s.sum() - 1) < 1e-9 and np.all(s > 0)
except Exception:
    ok_stat = False
check('stationary(P) satisfies pi P = pi and sums to 1', ok_stat)
check_hash('hidden: the stationary distribution', (np.asarray(pi_inf, float) if pi_inf is not None else None), 'f9d787133ce4acbc')
try:
    P2 = np.array([[0.9, 0.1], [0.5, 0.5]])
    ok2 = np.allclose(stationary(P2), [5 / 6, 1 / 6])
except Exception:
    ok2 = False
check('two-state weather chain gives (5/6, 1/6)', ok2)
try:
    far = propagate(P, [0., 0., 1.], 60)
    ok_conv = np.allclose(far, pi_inf, atol=1e-6)
except Exception:
    ok_conv = False
check('propagating 60 steps from any start converges to the stationary distribution', ok_conv)

## Exercise 12 — Mini-project — multi-feature linear regression with your own solver  `★★★`

The data cell generates 200 synthetic apartments with living area `size_m2`, number of `rooms`,
`age_years` and a sale price `price_k` in kEUR. The price was generated as
$50 + 2.5\,\text{size} + 15\,\text{rooms} - 1.2\,\text{age} + \varepsilon$ with
$\varepsilon \sim \mathcal{N}(0, 25^2)$ — pretend you do not know that.

Build an ordinary least-squares model of price from the three features **using only NumPy**:

1. Assemble the design matrix $X$ (with an intercept column) and target $\mathbf{y}$; report the
   shape and rank of $X$ and the condition numbers $\kappa(X)$ and $\kappa(X^\top X)$.
2. Fit the coefficients three ways — normal equations with `np.linalg.solve`, QR
   ($R\hat{\mathbf{w}} = Q^\top\mathbf{y}$) and `np.linalg.lstsq` — and confirm they agree.
3. Interpret every coefficient in words with its units (kEUR per m², per room, per year), and
   compare with the generating values.
4. Compute the fitted values via the hat matrix, the residual sum of squares, the residual
   standard deviation $\sqrt{\mathrm{RSS}/(n - p)}$ and $R^2 = 1 - \mathrm{RSS}/\sum_i (y_i - \bar y)^2$;
   check that the residuals are orthogonal to every column.
5. Verify against `sklearn.linear_model.LinearRegression` (`coef_`, `intercept_`).
6. Add a fourth feature `size_sqft = size_m2 * 10.764` and show what happens to the rank,
   the condition number and the coefficients — then explain what you see in one sentence.
7. Plot predicted versus actual prices (title, axis labels, a 45° reference line).

**Rubric — you have mastered this when you can:**

* The design matrix has an intercept column, shape `(200, 4)`, rank 4, and you report both condition numbers and explain why $\kappa(X^\top X) = \kappa(X)^2$.
* Normal equations, QR and `lstsq` agree to at least 8 significant digits and match scikit-learn's `coef_`/`intercept_`.
* Each coefficient is stated with units and compared with the true values $(50, 2.5, 15, -1.2)$; you note that `rooms` is correlated with `size_m2` ($\rho \approx 0.77$) and say what that does to the precision of those two coefficients.
* Residuals satisfy $X^\top\mathbf{r} \approx \mathbf{0}$; the residual standard deviation is close to the noise level 25 and $R^2$ is reported.
* Adding `size_sqft` leaves the rank at 4 although there are now 5 columns and makes $\kappa$ explode; you note that `np.linalg.solve` on the normal equations may raise, return enormous numbers or return a plausible-looking vector whose split between the two size columns is arbitrary, and you explain this as an exact collinearity (a null-space direction) rather than a numerical accident.
* The predicted-vs-actual plot is labelled and includes the $y = x$ reference line.

In [ ]:
# --- synthetic housing data (run this cell as given) -----------------------------------
g = np.random.default_rng(123)
n = 200
size_m2 = g.uniform(40, 160, n)                                   # living area
rooms = np.clip(np.round(size_m2 / 35 + g.normal(0, 0.7, n)), 1, 6)   # correlated with size
age_years = g.uniform(0, 60, n)
price_k = 50 + 2.5 * size_m2 + 15 * rooms - 1.2 * age_years + g.normal(0, 25, n)   # kEUR
# Your mini-project workspace — follow steps 1 to 7 of the prompt.

## Score

In [ ]:
summary()